# Retry Strategy (a job scheduler with pluggable backoff)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Onsite Loop, Concurrency, Math, OOP & Design Patterns · **Difficulty/Frequency:** Uncommon (3/10)

## Concepts

**What this problem is really testing:**
- The **strategy pattern** — separating a policy that varies from the loop that does not
- Whether you can spot that a "sequence" strategy does **not** need to be stateful
- Knowing why **backoff** exists at all, and what real retry loops add on top

**First-principles primer — what is each piece?**

- **Retry with backoff.** When a call fails, waiting before trying again. The *reason* matters: a failure usually means the far end is overloaded or briefly unavailable, so retrying **immediately** adds load to something already struggling. Backoff gives it room to recover. Retrying instantly, in a tight loop, is how a blip becomes an outage.
- **Strategy pattern.** One thing varies (how long to wait); everything else does not (call the job, catch the failure, count attempts, sleep, give up). Put the varying part behind a tiny interface and the loop never changes when you add a new policy.
- **The three curves.** Attempt 1, 2, 3, 4, 5 with `initial = 1`:

| Strategy | Delays | Total after 5 |
|---|---|---|
| Linear | 1, 1, 1, 1, 1 | 5 |
| Fibonacci | 1, 1, 2, 3, 5 | 12 |
| Exponential | 1, 2, 4, 8, 16 | 31 |

Fibonacci sits deliberately between the two — it backs off faster than linear but nowhere near as violently as exponential, which is why it shows up in protocols that want to be polite without giving up quickly.

**The insight the hints point at, and the official answer misses.**

> Hint 1: *"Think of each retry strategy as a **pure function** from the attempt number to a delay."*

Linear and exponential obviously are. Fibonacci **also** is — `fib(n)` depends only on `n`. The official answer keeps `prev`/`curr` as instance fields and mutates them per call, which:

- breaks if `next_delay` is called out of order, or twice for the same attempt,
- breaks if one strategy instance is **reused** for a second job,
- **corrupts silently** if two threads share an instance — and a job scheduler is exactly where that happens.

Computing `fib(n)` from `n` alone makes the strategy genuinely pure: reusable, thread-safe, testable in isolation. It costs an O(n) loop per call, and since `n` is a retry count (single digits), that is free. The notebook does both and asserts the difference.

**Simple worked example.** A job that fails twice then succeeds, with exponential backoff and `max_attempts = 4`:

```
attempt 1: run -> fails.  delay = 1 * 2^0 = 1s.  sleep
attempt 2: run -> fails.  delay = 1 * 2^1 = 2s.  sleep
attempt 3: run -> SUCCESS -> return True
```

Two delays for three attempts — the last attempt never sleeps, because there is nothing after it to wait for. That off-by-one is the classic bug: sleeping after the final failure just wastes the caller's time.

## Problem Statement

Build a `JobScheduler` that runs a job and retries it on failure, with the wait time decided by a **pluggable strategy**:

| Strategy | Delay before attempt *n* |
|---|---|
| **Linear** | a constant |
| **Exponential** | `initial × base^(n-1)` |
| **Fibonacci** | `initial × fib(n)` |

The scheduler owns the loop; the strategy only answers *how long to wait*.

### Approach 1 — Naive (the policy hard-coded inside the loop)

**Idea:** one scheduler with a `mode` string and an `if/elif` chain choosing the delay.

It works for three strategies and shows exactly why the pattern exists: every new policy means **editing the scheduler**, the delay logic is tangled with the retry logic, and you cannot test a curve without running a job. Fibonacci here also recomputes the sequence from scratch on every call.

**Time complexity:** O(m) attempts, but O(m²) total work for the Fibonacci branch (an O(n) recompute per attempt).

**Space complexity:** O(1).

In [ ]:
import time
from abc import ABC, abstractmethod
from typing import Callable, List, Optional, Tuple, Type


class NaiveScheduler:
    """Baseline: the delay policy is welded into the retry loop."""

    def __init__(self, max_attempts: int = 3, mode: str = "linear", initial: float = 1.0):
        self.max_attempts = max_attempts
        self.mode = mode
        self.initial = initial

    def _delay(self, attempt: int) -> float:
        if self.mode == "linear":
            return self.initial
        elif self.mode == "exponential":
            return self.initial * (2 ** (attempt - 1))
        elif self.mode == "fibonacci":
            a, b = 0, 1
            for _ in range(attempt):          # recomputed from scratch EVERY attempt
                a, b = b, a + b
            return self.initial * a
        raise ValueError(f"unknown mode: {self.mode}")   # a new policy means editing THIS class

    def run(self, job, sleep=time.sleep) -> bool:
        for attempt in range(1, self.max_attempts + 1):
            try:
                job()
                return True
            except Exception:
                if attempt == self.max_attempts:
                    return False
                sleep(self._delay(attempt))
        return False

### Approach 2 — Optimal (the strategy pattern, with *stateless* strategies)

**Idea:** one tiny interface — `next_delay(attempt) -> float` — and three implementations. The scheduler asks and waits; it never learns which policy it has.

**The design decision that matters:** every strategy is a **pure function of `attempt`**, Fibonacci included. Computing `fib(n)` from `n` rather than carrying `prev`/`curr` between calls means the object holds no mutable state, which buys three things at once:

- **Reusable** — one `ExponentialRetry()` can serve a thousand jobs.
- **Thread-safe** — no shared mutable state to interleave.
- **Testable in isolation** — you can assert the whole curve without running a job.

The iterative `fib` is O(n) per call, but `n` is a retry count in single digits. Trading a theoretically-worse inner loop for a genuinely pure interface is the right call here, and being able to *say* that is the point.

**Injecting `sleep`.** The scheduler takes the sleep function as a parameter. Real code sleeps; tests pass a recorder and run instantly while still asserting the exact delays. Without this, testing an exponential backoff means actually waiting 31 seconds.

**Time complexity:** O(m) attempts; O(1)–O(n) per delay lookup.

**Space complexity:** O(1).

In [ ]:
class RetryStrategy(ABC):
    """A pure function from attempt number to delay. No state, by design."""

    @abstractmethod
    def next_delay(self, attempt: int) -> float:
        """Seconds to wait before the given (1-based) attempt."""


class LinearRetry(RetryStrategy):
    def __init__(self, interval: float = 1.0) -> None:
        self.interval = interval

    def next_delay(self, attempt: int) -> float:
        return self.interval                       # 1, 1, 1, 1, ...


class ExponentialRetry(RetryStrategy):
    def __init__(self, base: float = 2.0, initial: float = 1.0) -> None:
        self.base = base
        self.initial = initial

    def next_delay(self, attempt: int) -> float:
        return self.initial * (self.base ** (attempt - 1))    # 1, 2, 4, 8, ...


class FibonacciRetry(RetryStrategy):
    """STATELESS: fib(n) is computed from n, so instances are reusable and thread-safe."""

    def __init__(self, initial: float = 1.0) -> None:
        self.initial = initial

    def next_delay(self, attempt: int) -> float:
        a, b = 0, 1
        for _ in range(attempt):                   # O(n), and n is a retry count - free
            a, b = b, a + b
        return self.initial * a                    # 1, 1, 2, 3, 5, 8, ...


class JobScheduler:
    """Owns the loop, the exception handling and the counting. Nothing else."""

    def __init__(self, max_attempts: int = 3) -> None:
        self.max_attempts = max_attempts

    def run(self, job: Callable[[], None], strategy: RetryStrategy,
            sleep: Callable[[float], None] = time.sleep) -> bool:
        for attempt in range(1, self.max_attempts + 1):
            try:
                job()
                return True                        # success: stop immediately
            except Exception:
                if attempt == self.max_attempts:
                    return False                   # do NOT sleep after the final failure
                sleep(strategy.next_delay(attempt))
        return False                               # max_attempts <= 0

### Approach 3 — Production-shaped (jitter, a cap, selective retry, observability)

**Idea:** four additions, each answering a follow-up and each solving a real failure mode.

- **A cap** (`max_delay`). Uncapped exponential backoff reaches 17 minutes by attempt 11. Every real client caps it — and the cap belongs in the **strategy**, since it is part of the curve, not part of the loop.
- **Jitter.** This is the one that actually matters in production. If 10,000 clients all fail at the same instant (a deploy, a network blip) and all back off by exactly 2 seconds, they **all retry at the same instant** — a synchronised thundering herd that re-kills the service just as it recovers. Randomising each delay spreads them out. The important detail: jitter is a **decorator around** a strategy, not baked into it, so the underlying curve stays deterministic and testable.
- **Selective retry** (`retry_on`). `except Exception` retries **bugs**. A `TypeError` from a mis-called job is not transient; retrying it three times with backoff just delays the failure and buries the stack trace. Naming the retryable exception types is what separates a real retry loop from a dangerous one.
- **An `on_failure` callback.** Logging inside the scheduler couples it to your logging stack. A callback hands each `(attempt, exception, delay)` to the caller and keeps the scheduler ignorant.

**Time complexity:** O(m).

**Space complexity:** O(1).

In [ ]:
import random


class CappedExponentialRetry(RetryStrategy):
    """Exponential, but never longer than max_delay. The cap is part of the CURVE."""

    def __init__(self, base: float = 2.0, initial: float = 1.0, max_delay: float = 60.0) -> None:
        self.base, self.initial, self.max_delay = base, initial, max_delay

    def next_delay(self, attempt: int) -> float:
        return min(self.initial * (self.base ** (attempt - 1)), self.max_delay)


class JitteredRetry(RetryStrategy):
    """DECORATES any strategy with randomness, leaving the wrapped curve deterministic."""

    def __init__(self, inner: RetryStrategy, factor: float = 0.5,
                 rng: Optional[random.Random] = None) -> None:
        if not 0.0 <= factor <= 1.0:
            raise ValueError("factor must be in [0, 1]")
        self.inner, self.factor = inner, factor
        self.rng = rng or random.Random()          # injectable, so tests are reproducible

    def next_delay(self, attempt: int) -> float:
        base = self.inner.next_delay(attempt)
        # Spread over [base*(1-factor), base]: same worst case, no synchronised herd.
        return base * (1 - self.factor * self.rng.random())


class ResilientScheduler:
    """Retries only what is worth retrying, and reports what it did."""

    def __init__(self, max_attempts: int = 3,
                 retry_on: Tuple[Type[BaseException], ...] = (Exception,)) -> None:
        self.max_attempts = max_attempts
        self.retry_on = retry_on                   # NOT bare `Exception` in production

    def run(self, job, strategy: RetryStrategy,
            sleep: Callable[[float], None] = time.sleep,
            on_failure: Optional[Callable[[int, BaseException, Optional[float]], None]] = None):
        last_error: Optional[BaseException] = None
        for attempt in range(1, self.max_attempts + 1):
            try:
                return True, job(), None
            except self.retry_on as exc:           # anything NOT listed propagates immediately
                last_error = exc
                delay = None if attempt == self.max_attempts else strategy.next_delay(attempt)
                if on_failure:
                    on_failure(attempt, exc, delay)   # observability without coupling
                if delay is None:
                    break
                sleep(delay)
        return False, None, last_error

## Verification

The delay curves are asserted exactly (no sleeping — the sleep function is injected and recorded), then the loop's counting, the off-by-one on the final attempt, and the statelessness that makes strategies reusable and thread-safe.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor


class Recorder:
    """Stands in for time.sleep: records the delays instead of waiting."""

    def __init__(self):
        self.delays: List[float] = []

    def __call__(self, seconds: float) -> None:
        assert seconds >= 0, f"a negative delay is nonsense: {seconds}"
        self.delays.append(seconds)


def failing_job(fail_times: int):
    """A job that fails `fail_times` times, then succeeds. Counts its own calls."""
    state = {"calls": 0}

    def job():
        state["calls"] += 1
        if state["calls"] <= fail_times:
            raise ConnectionError(f"attempt {state['calls']} failed")
        return "ok"

    return job, state


# --- The three curves, asserted exactly ---
assert [LinearRetry(1.0).next_delay(n) for n in range(1, 6)] == [1, 1, 1, 1, 1]
assert [ExponentialRetry(2.0, 1.0).next_delay(n) for n in range(1, 6)] == [1, 2, 4, 8, 16]
assert [FibonacciRetry(1.0).next_delay(n) for n in range(1, 9)] == [1, 1, 2, 3, 5, 8, 13, 21]

# Scaled by `initial`
assert [ExponentialRetry(3.0, 0.5).next_delay(n) for n in range(1, 4)] == [0.5, 1.5, 4.5]
assert [FibonacciRetry(2.0).next_delay(n) for n in range(1, 5)] == [2, 2, 4, 6]

# --- Statelessness: the SAME instance gives the same answer every time ---
fib = FibonacciRetry(1.0)
assert fib.next_delay(5) == 5
assert fib.next_delay(5) == 5, "a pure strategy must be idempotent"
assert fib.next_delay(3) == 2, "and must work when called OUT OF ORDER"
assert fib.next_delay(5) == 5, "...without the earlier calls corrupting it"

# One instance reused across several jobs must not drift
shared = FibonacciRetry(1.0)
for _ in range(3):
    rec = Recorder()
    job, _ = failing_job(fail_times=99)
    JobScheduler(max_attempts=5).run(job, shared, sleep=rec)
    assert rec.delays == [1, 1, 2, 3], f"a reused strategy must restart the curve: {rec.delays}"

# Thread safety: concurrent use of one instance must not interleave state
results: List[List[float]] = []
lock = threading.Lock()


def use_shared(_):
    got = [shared.next_delay(n) for n in range(1, 7)]
    with lock:
        results.append(got)


with ThreadPoolExecutor(max_workers=8) as ex:
    list(ex.map(use_shared, range(8)))
assert all(r == [1, 1, 2, 3, 5, 8] for r in results), "a stateless strategy is thread-safe"

# --- The scheduler's loop ---
rec = Recorder()
job, state = failing_job(fail_times=2)
assert JobScheduler(max_attempts=5).run(job, ExponentialRetry(), sleep=rec) is True
assert state["calls"] == 3, "3 calls: two failures then a success"
assert rec.delays == [1, 2], "2 sleeps for 3 attempts - the successful one never waits"

# A job that succeeds first time never sleeps at all
rec = Recorder()
job, state = failing_job(fail_times=0)
assert JobScheduler(max_attempts=5).run(job, ExponentialRetry(), sleep=rec) is True
assert state["calls"] == 1 and rec.delays == []

# THE off-by-one: exhausting the attempts must not sleep after the LAST failure
rec = Recorder()
job, state = failing_job(fail_times=99)
assert JobScheduler(max_attempts=3).run(job, LinearRetry(1.0), sleep=rec) is False
assert state["calls"] == 3, "exactly max_attempts calls"
assert rec.delays == [1, 1], f"max_attempts-1 sleeps, not {len(rec.delays)}"

# Degenerate attempt counts
for m in (0, -1):
    rec = Recorder()
    job, state = failing_job(fail_times=0)
    assert JobScheduler(max_attempts=m).run(job, LinearRetry(), sleep=rec) is False
    assert state["calls"] == 0 and rec.delays == []

rec = Recorder()
job, state = failing_job(fail_times=99)
assert JobScheduler(max_attempts=1).run(job, LinearRetry(), sleep=rec) is False
assert state["calls"] == 1 and rec.delays == [], "one attempt means no retries and no sleeping"

# The naive scheduler produces the same curves - it is only the DESIGN that differs
for mode, expected in [("linear", [1, 1, 1]), ("exponential", [1, 2, 4]),
                       ("fibonacci", [1, 1, 2])]:
    rec = Recorder()
    job, _ = failing_job(fail_times=99)
    NaiveScheduler(max_attempts=4, mode=mode).run(job, sleep=rec)
    assert rec.delays == expected, (mode, rec.delays)

# --- Capping ---
capped = CappedExponentialRetry(base=2.0, initial=1.0, max_delay=8.0)
assert [capped.next_delay(n) for n in range(1, 8)] == [1, 2, 4, 8, 8, 8, 8], "the cap holds"

# --- Jitter: bounded, random, and it leaves the wrapped curve untouched ---
inner = ExponentialRetry(2.0, 1.0)
jit = JitteredRetry(inner, factor=0.5, rng=random.Random(61))
for attempt in range(1, 6):
    base = inner.next_delay(attempt)
    for _ in range(200):
        d = jit.next_delay(attempt)
        assert base * 0.5 <= d <= base, f"jitter must stay within [0.5*base, base]: {d} vs {base}"
assert inner.next_delay(3) == 4, "the wrapped strategy is untouched and still deterministic"

# The whole point of jitter: the delays must actually DIFFER
samples = {JitteredRetry(inner, 0.5, random.Random(i)).next_delay(3) for i in range(50)}
assert len(samples) > 40, "jitter must spread clients out, not return one value"

# Jitter composes with the cap
both = JitteredRetry(CappedExponentialRetry(max_delay=8.0), factor=0.5,
                     rng=random.Random(3))
for attempt in range(1, 10):
    assert 0 <= both.next_delay(attempt) <= 8.0, "the cap survives jittering"

try:
    JitteredRetry(inner, factor=1.5)
except ValueError:
    pass
else:
    raise AssertionError("an out-of-range jitter factor must be rejected")

# --- Selective retry: a bug must NOT be retried ---
rec = Recorder()
calls = {"n": 0}


def buggy():
    calls["n"] += 1
    raise TypeError("this is a bug, not a transient failure")


sched = ResilientScheduler(max_attempts=5, retry_on=(ConnectionError, TimeoutError))
try:
    sched.run(buggy, ExponentialRetry(), sleep=rec)
except TypeError:
    pass
else:
    raise AssertionError("an exception outside retry_on must propagate immediately")
assert calls["n"] == 1, "a non-retryable error must be raised on the FIRST attempt"
assert rec.delays == [], "and must not sleep at all"

# ...while a listed exception IS retried
rec = Recorder()
job, state = failing_job(fail_times=2)          # raises ConnectionError
ok, value, err = sched.run(job, ExponentialRetry(), sleep=rec)
assert ok is True and value == "ok" and err is None
assert state["calls"] == 3 and rec.delays == [1, 2]

# --- The failure callback sees every attempt, and the delay that followed ---
seen: List[Tuple[int, str, Optional[float]]] = []
rec = Recorder()
job, _ = failing_job(fail_times=99)
ok, value, err = ResilientScheduler(max_attempts=3, retry_on=(ConnectionError,)).run(
    job, LinearRetry(2.0), sleep=rec,
    on_failure=lambda a, e, d: seen.append((a, type(e).__name__, d)),
)
assert ok is False and value is None and isinstance(err, ConnectionError)
assert seen == [(1, "ConnectionError", 2.0),
                (2, "ConnectionError", 2.0),
                (3, "ConnectionError", None)], seen
assert seen[-1][2] is None, "the final failure reports no delay - there is no next attempt"

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Where jitter belongs.** Implemented above as a **decorator** rather than a field on each strategy, and that placement is the answer the follow-up is fishing for: the underlying curve stays deterministic and unit-testable, while any strategy can be jittered by wrapping it. The variant here is "full jitter downward" (`[0.5·base, base]`); AWS's well-known analysis found *decorrelated* jitter — where each delay is drawn from a range based on the **previous** delay — spreads load even better, at the cost of reintroducing state. Worth naming the trade rather than asserting one is correct.
- **Where the cap belongs.** In the **strategy**. A ceiling on the delay is part of the shape of the curve, not part of "run the job and count attempts". Putting it in the scheduler would mean every scheduler needs a cap parameter even for linear backoff, which has no runaway to cap.
- **Going async.** `time.sleep` blocks the whole thread; `await asyncio.sleep(d)` yields, letting thousands of retrying jobs share one thread. The strategy interface **does not change at all** — it still returns a float — which is a nice demonstration that the policy was correctly separated. What changes is that `run` becomes a coroutine, and so every caller must `await` it: async is contagious upward through the call stack, and that is the real cost.
- **Retry only what is worth retrying.** Implemented as `retry_on`. The rule of thumb: retry **transient** failures (timeouts, connection resets, HTTP 429/503) and never retry **deterministic** ones (bad arguments, auth failures, 400s) — a bug retried three times is still a bug, just three times slower and with the original stack trace buried.
- **Idempotency — the thing this design cannot fix.** A retry assumes the failed attempt had **no effect**. If the job was "charge the customer" and the failure was a *timeout*, the charge may well have succeeded and only the response was lost. Retrying charges them twice. The fix is not in the retry loop at all: the *job* must carry an idempotency key so the server can recognise and discard the duplicate. Raising this unprompted is the strongest thing you can say about retries, because it is the failure mode that reaches production.
- **Retry storms and circuit breakers.** Backoff protects a struggling service from one client; it does nothing about ten thousand of them. A **circuit breaker** is the complement: after N consecutive failures, stop calling entirely for a cooldown period, then let a single probe through. Backoff spaces out *your* retries; a breaker stops them altogether when the far end is clearly down.

## Empirical complexity check

There is nothing asymptotically interesting in the retry loop itself — it is O(m) attempts. What *is* worth measuring is the difference the Concepts section claimed: the **stateless** Fibonacci recomputes `fib(n)` on every call, so a scheduler that runs to attempt m does O(m²) total work, while the stateful version does O(m).

The benchmark asks whether that theoretical difference matters at realistic retry counts.

| Growth as the attempt count doubles | What it means |
|---|---|
| ~4x | quadratic — the stateless recompute, as predicted |
| ~2x | linear — the stateful incremental version |

The point is the **absolute** numbers: at any plausible retry count (single digits), both are microseconds, which is why paying O(m²) to get a pure, thread-safe, reusable strategy is the right trade.

*(The sizes stop at 1000 for a concrete reason: Python integers are arbitrary-precision, but `float` is not. `fib(2000)` is about 4x10^417, and multiplying it by the float `initial` raises `OverflowError: int too large to convert to float`. A real retry loop caps the delay long before this — see `CappedExponentialRetry` — which is another argument for the cap belonging in the strategy.)*

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark


class StatefulFibonacciRetry(RetryStrategy):
    """The official answer's version: O(1) per call, but NOT reusable or thread-safe."""

    def __init__(self, initial: float = 1.0) -> None:
        self.initial, self.prev, self.curr = initial, 0, 1

    def next_delay(self, attempt: int) -> float:
        if attempt == 1:
            self.prev, self.curr = 0, 1
            return self.initial
        self.prev, self.curr = self.curr, self.prev + self.curr
        return self.initial * self.curr


def make_attempts(m):
    return (m,)


def run_stateless(m):
    s = FibonacciRetry()
    for attempt in range(1, m + 1):
        s.next_delay(attempt)            # O(n) each => O(m^2) total


def run_stateful(m):
    s = StatefulFibonacciRetry()
    for attempt in range(1, m + 1):
        s.next_delay(attempt)            # O(1) each => O(m) total


benchmark(
    {"stateless fib (pure, O(m^2) total)": run_stateless,
     "stateful fib (O(m) total, not reusable)": run_stateful},
    make_attempts,
    sizes=[125, 250, 500, 1000],
    repeats=2,
)

# Both produce the identical curve - the difference is design, not behaviour.
stateless = FibonacciRetry(1.0)
stateful = StatefulFibonacciRetry(1.0)
assert [stateless.next_delay(n) for n in range(1, 11)] == \
       [stateful.next_delay(n) for n in range(1, 11)]
print("\nBoth Fibonacci strategies produce the same curve; only their reusability differs.")

## Patterns learned

- **Separate what varies from what does not.** The loop (call, catch, count, sleep, give up) never changes; only the delay curve does. Put the curve behind a one-method interface and adding a strategy never touches the scheduler. That is the strategy pattern, and this is its textbook case.
- **Prefer a pure function to remembered state.** Fibonacci *looks* like it needs `prev`/`curr`, but `fib(n)` depends only on `n`. Dropping the state makes the object reusable, thread-safe and testable in isolation — worth far more than the O(1)-vs-O(n) inner loop it costs at realistic sizes.
- **Inject the clock.** Passing `sleep` as a parameter turns a 31-second test into a microsecond one, and lets you assert the *exact* delays instead of hoping. Anything your code waits on, randomises with, or reads the time from should be injectable.
- **Decorate rather than complicate.** Jitter wraps a strategy instead of becoming a field inside every strategy. The wrapped curve stays deterministic, and any future strategy gets jitter for free.
- **Never sleep after the last attempt.** There is nothing to wait for. `max_attempts` attempts means `max_attempts - 1` sleeps, and getting this wrong wastes the caller's time on every exhausted retry.
- **`except Exception` retries bugs.** Name the transient exception types. Retrying a `TypeError` three times with backoff turns a clear stack trace into a slow, confusing one.
- **Backoff exists to protect the far end, and jitter to protect it from your peers.** Immediate retries turn a blip into an outage; synchronised retries re-kill a service the moment it recovers.
- **A retry assumes the failed attempt had no effect.** When it might have (a timeout on a write), the answer is an idempotency key on the *job*, not anything the retry loop can do.